[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/electric_fields.ipynb)

# Electric charges and fields full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the level of the course's worked problems) and **tier 3** stretch problems (run a formula
backwards, combine two ideas, spot the trap). Each tier is harder than the last.

This notebook also has the course's **simulations and sums**: field lines traced a small step at a time, a dipole's
exact field against the far-away formula, the flux through a box added up tile by tile, a charged shell added up ring
by ring, and the smoke plates. You can move four of them yourself: the **second charge** of a field-line picture, the
**position of a charge** near a closed box, and the **plates' charge** and the **speck's electrons**.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Give the answer
in the unit the question asks for, to two decimals or more. Big and small numbers in E notation: `8.99e9` for
8.99 × 10⁹, `1.6e-19` for 1.6 × 10⁻¹⁹ (`8.99*10^9` works too). You can type the arithmetic: `8.99e9*15e-12/0.25`,
`sqrt(3.6/8.99e9)*0.05*1e6`, `500*0.4*cos(pi/3)`. Numbers without thousands separators: `28768`, not `28,768`. For
several answers, separate them with commas, in the order the question lists them (the box's label shows the order). You
can also put your answer between the quotes in a cell's last line, `practice("P1.4", "")`, and run the cell: it checks
straight away.

**The cheat sheet.** Every formula and constant from the course on two pages:
[electric charges and fields cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/electric_fields_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the constants, the simulator and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, IntSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # charges and lines in chalk, labels dim, ghost lines
SKY, MINT, CORAL, VIOLET, GOLD = "#6fb6bb", "#a0c27c", "#e3896b", "#b49ad6", "#e8c56b"
# velocity SKY · acceleration MINT · force and field CORAL · charges and lines CHALK · the one mark GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="simulated · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- the constants (the course's rounded values; the same in every question) ----
K = 8.99e9               # N m²/C², Coulomb's constant k
EPS0 = 8.85e-12          # C²/(N m²), epsilon nought (1/(4π ε₀) = 8.99 × 10⁹)
E = 1.60e-19             # C, the charge of one electron (its size)
ME = 9.11e-31            # kg, an electron's mass
g = 9.8                  # m/s²
E_SPARK = 3.0e6          # N/C: dry air at sea level sparks at about this field
ETA_AIR = 1.82e-5        # Pa s, air's viscosity at 20 °C (the smoke speck's drift)

# ---- the course's simulator: fourth-order Runge–Kutta, a fixed step at a time ----
def simulate(deriv, state, t_end, dt=1e-3, stop=None):
    """deriv(t, s) -> ds/dt as a tuple; state a tuple of numbers. Returns the list of (t, state) samples from 0 to
    t_end, or up to the first step where stop(t, s) is True."""
    t, s = 0.0, tuple(float(v) for v in state)
    out = [(t, s)]
    for _ in range(int(round(t_end / dt))):
        k1 = deriv(t, s)
        k2 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k1)))
        k3 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k2)))
        k4 = deriv(t + dt, tuple(a + dt * b for a, b in zip(s, k3)))
        s = tuple(a + dt / 6 * (b1 + 2 * b2 + 2 * b3 + b4) for a, b1, b2, b3, b4 in zip(s, k1, k2, k3, k4))
        t = t + dt
        out.append((t, s))
        if stop is not None and stop(t, s): return out
    return out

def close(a, b, rel=1e-4):
    """True when a simulated or summed value matches the formula (to a ten-thousandth by default)."""
    return abs(a - b) <= rel * max(abs(a), abs(b))

# ---- the course's formulas ----
def coulomb(q1, q2, r): return K * q1 * q2 / r ** 2                      # F = k q₁q₂/r²
def field_point(Q, r): return K * Q / r ** 2                             # E = kQ/r²
def flux(Ef, A, theta_deg): return Ef * A * math.cos(math.radians(theta_deg))   # Φ = E A cos θ
def dipole_axis_exact(q, a, r): return 4 * K * q * a * r / (r * r - a * a) ** 2
def dipole_axis(p, r): return 2 * K * p / r ** 3                          # far away, on the axis
def dipole_eq(p, r): return K * p / r ** 3                                # far away, on the equator
def torque(p, Ef, theta_deg): return p * Ef * math.sin(math.radians(theta_deg))
def gauss(q): return q / EPS0                                             # Φ = q_inside/ε₀
def wire(lam, r): return lam / (2 * math.pi * EPS0 * r)                   # E = λ/(2π ε₀ r)
def sheet(sigma): return sigma / (2 * EPS0)                               # E = σ/2ε₀
def plates(sigma): return sigma / EPS0                                    # σ/ε₀ between two plates ±σ

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "cos": math.cos, "sin": math.sin, "acos": math.acos}

def calc(text):
    """Plain arithmetic only: numbers (8.99e9 too), + − × ÷, powers, brackets, sqrt, cos, sin, acos (radians) and pi."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id in ("pi", "π"): return math.pi
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = (r"\s*(N m²/C|N m\^2/C|N m2/C|N/C|N m|C m|µC|uC|nC|C|N|m/s²|m/s\^2|m/s2|cm/s|m/s|cm|mm|m|s|%|°|degrees?|"
        r"times)\s*$")

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    s = s.replace("√", "sqrt")
    t = re.sub(UNIT, "", s).strip()
    try: return calc(t)
    except Exception: return s

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the course's questions and answers) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Electric charge',
     "q": 'A balloon carries −48 nC. How many extra electrons is that, in units of 10¹¹?',
     "parts": ['extra electrons (× 10¹¹)'],
     "answer": [3.0], "tol": [0.005], "function": '/',
     "hint": 'n = q ÷ e, with e = 1.60e-19',
     "working": 'n = |q|/e = 48 × 10⁻⁹ / 1.60 × 10⁻¹⁹ = 3.0000e+11 = 3.0000 × 10¹¹'},
    {"id": 'P1.2', "tier": 1, "group": 'Electric charge',
     "q": 'Two identical metal spheres carry +7 nC and −3 nC. They touch and are pulled apart. What is the charge on each, in nC?',
     "parts": ['charge on each (nC)'],
     "answer": [2.0], "tol": [0.005], "function": '—',
     "hint": 'charge is never made or destroyed: add, then share',
     "working": 'Total +4 nC, kept, shared equally: +2 nC each'},
    {"id": 'P1.3', "tier": 1, "group": "Coulomb's law",
     "q": 'Charges of +3 µC and +5 µC sit 0.5 m apart. What is the force between them, in newtons?',
     "parts": ['force (N)'],
     "answer": [0.5394], "tol": [0.005], "function": '—',
     "hint": 'F = k q1 q2 ÷ r², k = 8.99e9',
     "working": 'F = k q₁q₂/r² = 8.99 × 10⁹ × 15 × 10⁻¹²/0.25 = 0.5394 N'},
    {"id": 'P1.4', "tier": 1, "group": "Coulomb's law",
     "q": 'Two charges push apart with 36 N. Both charges are doubled and the distance is tripled. What is the new force, in newtons?',
     "parts": ['new force (N)'],
     "answer": [16.0], "tol": [0.005], "function": '—',
     "hint": 'the product of the charges on top, distance squared below',
     "working": '× 2 × 2 for the charges, ÷ 3² for the distance: 36 × 4/9 = 16 N'},
    {"id": 'P1.5', "tier": 1, "group": 'The electric field',
     "q": 'What is the electric field 20 cm from a +5 nC point charge, in newtons per coulomb?',
     "parts": ['field (N/C)'],
     "answer": [1123.7499999999998], "tol": [0.005], "function": '—',
     "hint": 'E = k Q ÷ r²',
     "working": 'E = kQ/r² = 8.99 × 10⁹ × 5 × 10⁻⁹/0.04 = 1123.7500 N/C'},
    {"id": 'P1.6', "tier": 1, "group": 'The electric field',
     "q": 'A proton (charge 1.60 × 10⁻¹⁹ C) sits in a field of 2.5 × 10⁴ N/C. What force does it feel, in units of 10⁻¹⁵ N?',
     "parts": ['force (× 10⁻¹⁵ N)'],
     "answer": [3.9999999999999996], "tol": [0.005], "function": '—',
     "hint": 'F = q E',
     "working": 'F = qE = 1.60 × 10⁻¹⁹ × 2.5 × 10⁴ = 4.0 × 10⁻¹⁵ N'},
    {"id": 'P1.7', "tier": 1, "group": 'Electric flux',
     "q": "A flat surface of area 0.4 m² sits in a uniform field of 500 N/C. The field makes 60° with the surface's area vector. What is the flux, in N m²/C?",
     "parts": ['flux (N m²/C)'],
     "answer": [100.00000000000003], "tol": [0.005], "function": 'cos',
     "hint": 'Φ = E A cos θ; in Python math.cos(math.radians(60))',
     "working": 'Φ = E A cos θ = 500 × 0.4 × cos 60° = 100.0000 N m²/C'},
    {"id": 'P1.8', "tier": 1, "group": 'The electric dipole',
     "q": 'Charges of +3 nC and −3 nC sit 2 mm apart. What is the dipole moment, in units of 10⁻¹² C m?',
     "parts": ['dipole moment (× 10⁻¹² C m)'],
     "answer": [6.0], "tol": [0.005], "function": '—',
     "hint": 'p = q × (the distance between the charges)',
     "working": 'p = q × 2a = 3 × 10⁻⁹ × 2 × 10⁻³ = 6 × 10⁻¹² C m'},
    {"id": 'P1.9', "tier": 1, "group": "Gauss's law",
     "q": 'A closed surface surrounds a charge of +4.0 nC. What is the total flux out of it, in N m²/C?',
     "parts": ['flux (N m²/C)'],
     "answer": [451.97740112994353], "tol": [0.005], "function": '—',
     "hint": 'Φ = q_inside ÷ ε₀, ε₀ = 8.85e-12',
     "working": 'Φ = q/ε₀ = 4.0 × 10⁻⁹/8.85 × 10⁻¹² = 451.9774 N m²/C'},
    {"id": 'P1.10', "tier": 1, "group": "Gauss's law at work",
     "q": 'A large flat sheet carries 3.0 µC per square metre. What is the field near it, in units of 10⁵ N/C?',
     "parts": ['field (× 10⁵ N/C)'],
     "answer": [1.694915254237288], "tol": [0.005], "function": '—',
     "hint": 'E = σ ÷ (2 ε₀)',
     "working": 'E = σ/2ε₀ = 3.0 × 10⁻⁶/(2 × 8.85 × 10⁻¹²) = 169491.5 N/C = 1.6949 × 10⁵ N/C'},
    {"id": 'P2.1', "tier": 2, "group": 'Electric charge',
     "q": 'A metal sphere with +12 nC touches an identical neutral sphere; they part. Then the second sphere touches a third identical neutral sphere. What charge does the third sphere end with, in nC?',
     "parts": ['charge on the third sphere (nC)'],
     "answer": [3.0], "tol": [0.005], "function": '—',
     "hint": 'share equally at each touch',
     "working": '12 → 6 and 6; then 6 → 3 and 3: the third has +3 nC (the total, 12 nC, is kept: 6 + 3 + 3)'},
    {"id": 'P2.2', "tier": 2, "group": "Coulomb's law",
     "q": 'On a line: +2 µC at 0, +3 µC at 0.2 m, −4 µC at 0.5 m. What is the total force on the +3 µC charge, in newtons (positive means towards the −4 µC)?',
     "parts": ['total force (N)'],
     "answer": [2.5471666666666666], "tol": [0.005], "function": '—',
     "hint": 'work out each pair alone, then add with signs',
     "working": 'the +2 µC pushes it along +: 1.3485 N; the −4 µC pulls it along +: 1.1987 N; total 2.5472 N'},
    {"id": 'P2.3', "tier": 2, "group": "Coulomb's law",
     "q": 'Two equal charges 5 cm apart push each other with 3.6 N. How big is each charge, in µC?',
     "parts": ['each charge (µC)'],
     "answer": [1.000556018947605], "tol": [0.005], "function": 'sqrt',
     "hint": 'F = k q² ÷ r², so q = r × sqrt(F ÷ k)',
     "working": 'q = r √(F/k) = 0.05 × √(3.6/8.99 × 10⁹) = 1.0006e-06 C = 1.0006 µC'},
    {"id": 'P2.4', "tier": 2, "group": 'The electric field',
     "q": 'Charges of +6 nC and −2 nC sit 10 cm apart. What is the field at the midpoint, in newtons per coulomb?',
     "parts": ['field at the midpoint (N/C)'],
     "answer": [28767.999999999996], "tol": [0.005], "function": '—',
     "hint": 'both arrows point the same way: add the sizes',
     "working": 'both fields point towards the −2 nC: k(6 + 2) × 10⁻⁹/0.05² = 28768.0000 N/C'},
    {"id": 'P2.5', "tier": 2, "group": 'The electric field',
     "q": 'An electron (charge 1.60 × 10⁻¹⁹ C, mass 9.11 × 10⁻³¹ kg) sits in a field of 1.2 × 10⁴ N/C. What is its acceleration, in units of 10¹⁵ m/s²?',
     "parts": ['acceleration (× 10¹⁵ m/s²)'],
     "answer": [2.1075740944017562], "tol": [0.005], "function": '—',
     "hint": 'a = F ÷ m and F = e E',
     "working": 'a = eE/m = 1.60 × 10⁻¹⁹ × 1.2 × 10⁴/9.11 × 10⁻³¹ = 2.1076e+15 m/s²'},
    {"id": 'P2.6', "tier": 2, "group": 'Electric flux',
     "q": 'A cube 0.2 m on a side sits in a uniform field of 800 N/C pointing along +x. What is the flux through the face where the field leaves, through the face where it enters, and through the whole cube, in N m²/C?',
     "parts": ['leaving face', 'entering face', 'whole cube'],
     "answer": [32.0, -32.0, 0.0], "tol": [0.005, 0.005, 0.005], "function": '—',
     "hint": 'area vectors point outwards; in counts as negative',
     "working": 'leaving face: 800 × 0.04 = 32; entering face: −32 (the area vector points out, against the field); the four side faces: 0; the whole cube: 0'},
    {"id": 'P2.7', "tier": 2, "group": 'The electric dipole',
     "q": 'A dipole has p = 5 × 10⁻¹² C m. Using the far-away formulas, what is its field 10 cm away on its axis, and 10 cm away on its equator, in N/C?',
     "parts": ['on the axis (N/C)', 'on the equator (N/C)'],
     "answer": [89.89999999999998, 44.94999999999999], "tol": [0.005, 0.005], "function": '—',
     "hint": 'axis 2kp ÷ r³, equator k p ÷ r³',
     "working": 'axis 2kp/r³ = 89.9000 N/C; equator kp/r³ = 44.9500 N/C'},
    {"id": 'P2.8', "tier": 2, "group": 'The electric dipole',
     "q": 'The same dipole (p = 5 × 10⁻¹² C m) sits in a uniform field of 2 × 10⁴ N/C at 30° to it. What is the torque, in units of 10⁻⁸ N m?',
     "parts": ['torque (× 10⁻⁸ N m)'],
     "answer": [4.999999999999999], "tol": [0.005], "function": 'sin',
     "hint": 'τ = p E sin θ',
     "working": 'τ = pE sin θ = 5 × 10⁻¹² × 2 × 10⁴ × 0.5 = 5.0000e-08 N m'},
    {"id": 'P2.9', "tier": 2, "group": "Gauss's law",
     "q": 'A +3 µC charge sits at the centre of a cube. What is the flux through one face, in units of 10⁴ N m²/C?',
     "parts": ['flux through one face (× 10⁴ N m²/C)'],
     "answer": [5.649717514124293], "tol": [0.005], "function": '—',
     "hint": 'the whole flux is q ÷ ε₀; symmetry splits it six ways',
     "working": 'the six faces are alike: Φ = q/(6ε₀) = 3 × 10⁻⁶/(6 × 8.85 × 10⁻¹²) = 56497.2 N m²/C'},
    {"id": 'P2.10', "tier": 2, "group": "Gauss's law at work",
     "q": 'A long straight wire carries 1.5 µC per metre. What is the field 10 cm from it, in units of 10⁵ N/C?',
     "parts": ['field (× 10⁵ N/C)'],
     "answer": [2.6975414083372087], "tol": [0.005], "function": 'pi',
     "hint": 'E = λ ÷ (2 π ε₀ r)',
     "working": 'E = λ/(2π ε₀ r) = 1.5 × 10⁻⁶/(2π × 8.85 × 10⁻¹² × 0.1) = 269754.1 N/C'},
    {"id": 'P3.1', "tier": 3, "group": 'Electric charge',
     "q": 'A balloon carries −100 nC. Over one minute it loses 1 % of its extra electrons to water molecules in the air. How many electrons moved off, in units of 10⁹? (Where did the charge go?)',
     "parts": ['electrons moved (× 10⁹)'],
     "answer": [6.25], "tol": [0.005], "function": '—',
     "hint": 'count the electrons first: q ÷ e',
     "working": '6.25 × 10¹¹ × 1 % = 6.2500e+09: they moved to the air; none was destroyed'},
    {"id": 'P3.2', "tier": 3, "group": "Coulomb's law",
     "q": '+9 µC sits at 0 and −1 µC at 0.4 m. Where on the line can a third charge feel no force? Give its position in metres.',
     "parts": ['position (m)'],
     "answer": [0.6], "tol": [0.005], "function": '—',
     "hint": 'outside, near the smaller charge; take square roots',
     "working": 'Not between them (both pushes point the same way there). Beyond the small charge: 9/x² = 1/(x − 0.4)² → 3(x − 0.4) = x → x = 0.6 m'},
    {"id": 'P3.3', "tier": 3, "group": 'The electric field',
     "q": '+4 nC and +9 nC sit 50 cm apart. How far from the +4 nC charge is the field zero, in cm?',
     "parts": ['distance from the +4 nC (cm)'],
     "answer": [20.0], "tol": [0.005], "function": '—',
     "hint": 'between two like charges; take square roots',
     "working": '4/x² = 9/(0.5 − x)² → 2(0.5 − x) = 3x → x = 0.2 m = 20 cm'},
    {"id": 'P3.4', "tier": 3, "group": 'The electric dipole',
     "q": 'A dipole of ±1 nC, 2 mm apart. At a point on its axis 3 mm from its centre, by what percent does the far-away formula 2kp/r³ fall short of the exact field?',
     "parts": ['how far short (%)'],
     "answer": [20.987654320987648], "tol": [0.005], "function": '—',
     "hint": 'r is only three times a: the far formula is poor this close',
     "working": 'exact 4kqar/(r² − a²)² = 1685625.00 N/C; 2kp/r³ = 1331851.85 N/C: short by 20.9877 %'},
    {"id": 'P3.5', "tier": 3, "group": "Gauss's law",
     "q": 'A closed surface holds +3 µC and −5 µC; a +7 µC charge sits just outside it. What is the total flux out of the surface, in units of 10⁵ N m²/C?',
     "parts": ['flux (× 10⁵ N m²/C)'],
     "answer": [-2.259887005649717], "tol": [0.005], "function": '—',
     "hint": 'the +7 µC is outside: does it count?',
     "working": 'only the charge inside counts: (3 − 5) µC/ε₀ = -225988.7 N m²/C (the outside charge changes the field on the surface, not the flux)'},
    {"id": 'P3.6', "tier": 3, "group": "Gauss's law at work",
     "q": 'Two large parallel plates carry +1.77 µC and −1.77 µC per square metre. What is the field between them and outside them, in units of 10⁵ N/C?',
     "parts": ['between (× 10⁵ N/C)', 'outside (× 10⁵ N/C)'],
     "answer": [2.0, 0.0], "tol": [0.005, 0.005], "function": '—',
     "hint": 'each sheet gives σ ÷ 2ε₀; add as arrows',
     "working": "between: σ/ε₀ = 200000.0 N/C; outside the two sheets' fields cancel: 0"},
    {"id": 'P3.7', "tier": 3, "group": "Gauss's law at work",
     "q": 'Air sparks at about 3.0 × 10⁶ N/C. What is the largest charge a metal ball 5 cm in radius can hold in air, in nC? What is the field at its centre (N/C)?',
     "parts": ['largest charge (nC)', 'field at the centre (N/C)'],
     "answer": [834.2602892102339, 0.0], "tol": [0.005, 0.005], "function": '—',
     "hint": 'outside, a ball acts as a point at its centre',
     "working": 'at the surface kQ/R² = 3 × 10⁶ → Q = 3 × 10⁶ × 0.05²/8.99 × 10⁹ = 834.2603 nC; inside the shell of charge, 0'},
    {"id": 'P3.8', "tier": 3, "group": "Gauss's law at work",
     "q": 'A speck of smoke (mass 5.24 × 10⁻¹⁶ kg) carries 200 extra electrons between plates holding ±1 µC per square metre. How many times its weight is the electric force on it?',
     "parts": ['force ÷ weight'],
     "answer": [704.1243201899724], "tol": [0.005], "function": '—',
     "hint": 'field between plates σ ÷ ε₀, then F = q E',
     "working": 'E = σ/ε₀ = 112994.4 N/C; F = 200 × 1.60 × 10⁻¹⁹ × E = 3.6158e-12 N; weight 5.24 × 10⁻¹⁶ × 9.8; ratio 704.1243'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g_, w, t) for g_, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else "✓ all right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
            print(textwrap.fill(f"Hint: {q['hint']}", 110))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g_, w, t) for g_, w, t in zip(got, answers(k), Q[k]["tol"]))

assert round(1 / (4 * math.pi * EPS0) / 1e9, 2) == 8.99
print("Ready: k =", K, "N m²/C²; 1/(4π ε₀) =", f"{1 / (4 * math.pi * EPS0):.4e}", "N m²/C²")

## Tier 1 · Warm-up

One idea, one step: counting electrons, sharing charge, Coulomb's law, the inverse square, the field of a point
charge, the force in a field, flux through a tilted surface, a dipole moment, Gauss's law, and a charged sheet.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each formula from the course is one short function in the first cell: `coulomb(q1, q2, r)`, `field_point(Q, r)`,
`flux(E, A, theta)`, `gauss(q)`, `sheet(sigma)` and the rest. Work in coulombs, metres and newtons (nC is `1e-9`, µC is
`1e-6`, cm is `0.01`), then change the unit at the end. Angles go into `math.cos` in radians: `math.radians(60)` first.
Each line below computes one warm-up answer and checks it against the practice set.

In [ ]:
print(f"P1.3  F = {coulomb(3e-6, 5e-6, 0.5):.4f} N")
print(f"P1.5  E = {field_point(5e-9, 0.2):.2f} N/C;  P1.7  flux = {flux(500, 0.4, 60):.4f} N m²/C")
assert agrees("P1.1", [48e-9 / E / 1e11]) and agrees("P1.2", [(7 + -3) / 2])
assert agrees("P1.3", [coulomb(3e-6, 5e-6, 0.5)]) and agrees("P1.4", [36 * 2 * 2 / 3 ** 2])
assert agrees("P1.5", [field_point(5e-9, 0.2)]) and agrees("P1.6", [E * 2.5e4 * 1e15])
assert agrees("P1.7", [flux(500, 0.4, 60)]) and agrees("P1.8", [3e-9 * 2e-3 * 1e12])
assert agrees("P1.9", [gauss(4e-9)]) and agrees("P1.10", [sheet(3e-6) / 1e5])
print("Every tier 1 answer, worked out in Python, matches the practice set.")

## Field lines, traced

A field line is drawn the way the course draws it: start just off a positive charge, take a small step along the
field, look again, step again. It stops when it reaches a negative charge or leaves the picture. Twelve lines leave
every unit of positive charge. With +2 on the left and −1 on the right, 24 lines leave the +2 and only 12 end on the
−1: the other 12 go far away. Lines never cross, and they show the push on a small positive charge, not its path.
Slide the second charge, from −2 to +2, and run it again.

In [ ]:
def efield(charges, x, y):
    ex = ey = 0.0
    for q, cx, cy in charges:
        dx, dy = x - cx, y - cy
        d3 = (dx * dx + dy * dy) ** 1.5
        ex += q * dx / d3; ey += q * dy / d3
    return ex, ey

def trace_line(charges, x0, y0, ds=0.01, length=60.0, box=1000.0, sink=0.05):
    """One field line (k = 1 units). Returns (points, end): end is the index of the negative charge it reaches, or None
    when it leaves the box. The step grows with the distance from the middle (the same curve, fewer steps far out)."""
    def f(t, s):
        ex, ey = efield(charges, s[0], s[1]); m = math.hypot(ex, ey)
        w = max(1.0, math.hypot(s[0], s[1]) / 2)
        return (w * ex / m, w * ey / m)
    def stop(t, s):
        if abs(s[0]) > box or abs(s[1]) > box: return True
        return any(q < 0 and math.hypot(s[0] - cx, s[1] - cy) < sink for q, cx, cy in charges)
    run = simulate(f, (x0, y0), length, ds, stop)
    xe, ye = run[-1][1]
    end = next((i for i, (q, cx, cy) in enumerate(charges) if q < 0 and math.hypot(xe - cx, ye - cy) < sink), None)
    return [p[1] for p in run], end

def field_lines_picture(second=-1.0, first=2.0, per_unit=12):
    charges = [(first, -1.0, 0.0), (second, 1.0, 0.0)]
    lines = []
    for i, (q, cx, cy) in enumerate(charges):
        if q <= 0: continue
        n = max(1, int(round(per_unit * q)))
        for k in range(n):
            th = 2 * math.pi * (k + 0.5) / n
            lines.append(trace_line(charges, cx + 0.05 * math.cos(th), cy + 0.05 * math.sin(th)))
    landed = sum(1 for _, e in lines if e == 1)
    fig, ax = plt.subplots(figsize=(9.5, 6.2))
    fig.subplots_adjust(left=0.06, right=0.98, top=0.88, bottom=0.08)
    for pts, e in lines:
        xs, ys = [p[0] for p in pts], [p[1] for p in pts]
        ax.plot(xs, ys, color=GOLD if e == 1 else CHALK, lw=1.4, alpha=0.95 if e == 1 else 0.7)
    for q, cx, cy in charges:
        if q == 0: continue
        ax.plot([cx], [cy], "o", ms=10 + 6 * abs(q), color=CORAL if q > 0 else SKY)
        ax.text(cx, cy + 0.32, f"{q:+.2g}", color=CHALK, ha="center")
    ax.set_xlim(-4, 4); ax.set_ylim(-2.8, 2.8); ax.set_aspect("equal"); ax.grid(False)
    ax.set_xticks([]); ax.set_yticks([]); [s_.set_visible(False) for s_ in ax.spines.values()]
    title = f"{len(lines)} lines leave the positive charges"
    if second < 0: title += f"; {landed} end on the {second:+.2g} charge (gold)"
    fig.suptitle(title, x=0.06, ha="left", fontsize=15, color=GOLD)
    footnote(fig, "traced step by step from the charges"); plt.show()
    return len(lines), landed

n_lines, n_landed = field_lines_picture(-1.0)
assert (n_lines, n_landed) == (24, 12)                   # lines follow the charge: half of the +2's lines reach the −1

In [ ]:
interact(field_lines_picture, second=FloatSlider(value=-1.0, min=-2.0, max=2.0, step=0.25, description="second q"),
         first=FloatSlider(value=2.0, min=0.5, max=3.0, step=0.5, description="first q"));

## Tier 2 · Core

Several steps each: charge shared twice, three charges on a line, two equal charges from their push, the field between
two charges, an electron's acceleration, the flux through every face of a cube, a dipole's field on its axis and its
equator, the torque on a dipole, the flux through one face of a cube, and a charged wire.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

Superposition is just a sum with signs: work out each pair alone, give it a sign for its direction, then add. Running
a formula backwards is algebra first, then one line: q = r √(F/k).

In [ ]:
assert agrees("P2.1", [12 / 2 / 2])
f_left, f_right = coulomb(2e-6, 3e-6, 0.2), coulomb(3e-6, 4e-6, 0.3)     # both push the +3 µC towards the −4 µC
print(f"P2.2  {f_left:.4f} N + {f_right:.4f} N = {f_left + f_right:.4f} N")
assert agrees("P2.2", [f_left + f_right]) and agrees("P2.3", [0.05 * math.sqrt(3.6 / K) * 1e6])
assert agrees("P2.4", [field_point(6e-9, 0.05) + field_point(2e-9, 0.05)]) and agrees("P2.5", [E * 1.2e4 / ME / 1e15])
assert agrees("P2.6", [800 * 0.2 ** 2, -800 * 0.2 ** 2, 0.0])
assert agrees("P2.7", [dipole_axis(5e-12, 0.1), dipole_eq(5e-12, 0.1)]) and agrees("P2.8", [torque(5e-12, 2e4, 30) * 1e8])
assert agrees("P2.9", [gauss(3e-6) / 6 / 1e4]) and agrees("P2.10", [wire(1.5e-6, 0.1) / 1e5])
print("Every tier 2 answer, worked out in Python, matches the practice set.")

## A dipole: the exact field against the far-away formula

On a dipole's axis the exact field is 4kqar/(r² − a²)²; far away it becomes 2kp/r³. The curve shows how far apart they
are as you move out: about 2 % at ten times a, about a fifth of a percent at thirty, and badly wrong close in.

In [ ]:
q_d, a_d = 1e-9, 1e-3                                                    # ±1 nC, 2 mm apart: p = 2 × 10⁻¹² C m
ns = np.linspace(1.5, 50, 300)
ratio = [dipole_axis_exact(q_d, a_d, n * a_d) / dipole_axis(q_d * 2 * a_d, n * a_d) for n in ns]
fig, ax = plt.subplots(figsize=(10, 4.2))
fig.subplots_adjust(left=0.09, right=0.98, top=0.84, bottom=0.17)
ax.plot(ns, ratio, color=CHALK)
for n, lab in ((10, "10a: 2 % more"), (30, "30a: 0.2 % more")):
    v = dipole_axis_exact(q_d, a_d, n * a_d) / dipole_axis(q_d * 2 * a_d, n * a_d)
    ax.plot([n], [v], "o", color=GOLD, ms=8); ax.annotate(lab, (n, v), (n + 1.5, v + 0.15), color=GOLD)
ax.set_ylim(0.9, 2.2); ax.set_xlabel("distance along the axis, in units of a"); ax.set_ylabel("exact ÷ far-away")
fig.suptitle("A dipole's field: the far-away formula is fine beyond about ten times a", x=0.09, ha="left", color=GOLD)
footnote(fig, "computed from the two charges' fields"); plt.show()
r10 = dipole_axis_exact(q_d, a_d, 10 * a_d) / dipole_axis(2e-12, 10 * a_d)
r30 = dipole_axis_exact(q_d, a_d, 30 * a_d) / dipole_axis(2e-12, 30 * a_d)
assert round(100 * (r10 - 1)) == 2 and round(100 * (r30 - 1), 1) == 0.2
assert close(K * q_d / (0.01 - a_d) ** 2 - K * q_d / (0.01 + a_d) ** 2, dipole_axis_exact(q_d, a_d, 0.01), 1e-12)

## Gauss's law, added up tile by tile

A closed, lopsided box (1 m by 0.8 m by 0.6 m). Each face is cut into small tiles; for each tile, the field of a point
charge times the tile's area, counted plus where the field leaves and minus where it enters. The total, divided by
q/ε₀, is 1 wherever the charge sits inside, and 0 the moment it leaves. Slide the charge along the box's long side:
the box runs from 0 to 1 m.

In [ ]:
def box_flux(q, pos, size=(1.0, 0.8, 0.6), n=80):
    a_, b_, c_ = size
    tot = 0.0
    for axis, (L1, L2), L0 in ((0, (b_, c_), a_), (1, (a_, c_), b_), (2, (a_, b_), c_)):
        h1, h2 = L1 / n, L2 / n
        u = (np.arange(n) + 0.5) * h1
        v = (np.arange(n) + 0.5) * h2
        U, V = np.meshgrid(u, v, indexing="ij")
        for face, sign in ((0.0, -1.0), (L0, 1.0)):
            pts = [U, V]; pts.insert(axis, np.full_like(U, face))
            d = [pts[i] - pos[i] for i in range(3)]
            d3 = (d[0] ** 2 + d[1] ** 2 + d[2] ** 2) ** 1.5
            tot += sign * np.sum(K * q * d[axis] / d3) * h1 * h2
    return tot

def flux_picture(x=0.5):
    q = 1e-6
    ratio = box_flux(q, (x, 0.4, 0.3)) / gauss(q)
    fig, ax = plt.subplots(figsize=(9, 3.6))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.8, bottom=0.2)
    ax.add_patch(plt.Rectangle((0, 0), 1.0, 0.6, fill=False, ec=DIM, lw=2))
    ax.plot([x], [0.3], "o", ms=14, color=CORAL); ax.text(x, 0.42, "+1 µC", color=CHALK, ha="center")
    ax.set_xlim(-0.2, 1.8); ax.set_ylim(-0.1, 0.75); ax.set_aspect("equal"); ax.grid(False); ax.set_yticks([])
    ax.set_xlabel("position along the box (m)")
    fig.suptitle(f"flux out ÷ (q/ε₀) = {ratio:.4f}", x=0.07, ha="left", fontsize=16, color=GOLD)
    footnote(fig, "added up tile by tile: 80 × 80 tiles a face"); plt.show()
    return ratio

assert abs(flux_picture(0.5) - 1) < 2e-3                                # inside: q/ε₀
assert abs(flux_picture(1.4)) < 2e-3                                    # outside: nothing

In [ ]:
interact(flux_picture, x=FloatSlider(value=0.5, min=0.1, max=1.7, step=0.05, description="charge x (m)"));

## Tier 3 · Stretch

Run a formula backwards, combine two ideas, spot the trap: where the electrons went, the no-force point beyond two
charges, the zero-field point between two, how poor the far-away dipole formula is close in, a charge outside a closed
surface, two plates, the most charge a metal ball can hold in air, and a speck of smoke between charged plates.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

Where two pushes balance, take square roots first: 9/x² = 1/(x − 0.4)² becomes 3(x − 0.4) = x. And Gauss's law only
counts what is inside: the +7 µC in P3.5 never enters the sum.

In [ ]:
assert agrees("P3.1", [0.01 * 100e-9 / E / 1e9]) and agrees("P3.2", [0.4 * 3 / 2]) and agrees("P3.3", [0.5 * 2 / 5 * 100])
ex3, ap3 = dipole_axis_exact(1e-9, 1e-3, 3e-3), dipole_axis(2e-12, 3e-3)
print(f"P3.4  exact {ex3:,.1f} N/C, far-away {ap3:,.1f} N/C: short by {100 * (ex3 - ap3) / ex3:.2f} %")
assert agrees("P3.4", [100 * (ex3 - ap3) / ex3]) and agrees("P3.5", [gauss(3e-6 - 5e-6) / 1e5])
assert agrees("P3.6", [plates(1.77e-6) / 1e5, 0.0]) and agrees("P3.7", [E_SPARK * 0.05 ** 2 / K * 1e9, 0.0])
assert agrees("P3.8", [200 * E * plates(1e-6) / (5.24e-16 * g)])
print("Every tier 3 answer, worked out in Python, matches the practice set.")

## A charged shell, added up ring by ring

A thin shell of radius 1 m carrying 1 nC, cut into 20,000 rings. At each distance from the centre the rings' pulls are
added. Outside, the total is kQ/r², as if all the charge sat at the centre. Inside, it is zero, everywhere, not only at
the centre: the near wall is closer but smaller, the far wall bigger by exactly the square of the distance.

In [ ]:
def shell_field(Q, Rs, rr, n=20000):
    th = (np.arange(n) + 0.5) * np.pi / n
    dq = Q / 2 * np.sin(th) * np.pi / n
    s2 = Rs * Rs + rr * rr - 2 * Rs * rr * np.cos(th)
    return float(np.sum(K * dq * (rr - Rs * np.cos(th)) / s2 ** 1.5))

rs = [0.0, 0.2, 0.4, 0.6, 0.8, 1.2, 1.5, 2.0, 3.0]
es = [shell_field(1e-9, 1.0, x) for x in rs]
fig, ax = plt.subplots(figsize=(10, 4.2))
fig.subplots_adjust(left=0.09, right=0.98, top=0.84, bottom=0.17)
xs = np.linspace(1.0, 3.2, 200)
ax.plot([0, 1], [0, 0], color=DIM, lw=2); ax.plot(xs, K * 1e-9 / xs ** 2, color=DIM, lw=2, label="kQ/r² outside, 0 inside")
ax.plot(rs, es, "o", color=GOLD, ms=9, label="added up ring by ring")
ax.axvline(1.0, color=GHOST, lw=1.4); ax.text(1.03, 9.5, "the shell", color=DIM)
ax.set_xlabel("distance from the centre (m)"); ax.set_ylabel("E (N/C)"); ax.legend(frameon=False)
fig.suptitle("Inside a charged shell, no field anywhere", x=0.09, ha="left", color=GOLD)
footnote(fig, "summed: 20,000 rings"); plt.show()
assert all(abs(v) < 1e-6 * K * 1e-9 for x, v in zip(rs, es) if x < 1)
assert all(close(v, K * 1e-9 / x ** 2, 1e-6) for x, v in zip(rs, es) if x > 1.1)

## The smoke plates: change the charge, watch the pull

The course's opening question. Two plates carry ±σ; between them the field is σ/ε₀, the same everywhere. A speck of
smoke 1 µm across, as dense as water, carries some extra electrons. The pull qE against its weight mg, and the steady
drift that air drag allows (Stokes's law, 6πηr per metre per second). Above about 3 × 10⁶ N/C, the air would spark.

In [ ]:
def plates_picture(sigma_uC=2.0, electrons=100):
    d, rho = 1.0e-6, 1000.0
    m = math.pi / 6 * d ** 3 * rho
    Ef = plates(sigma_uC * 1e-6)
    F, W = electrons * E * Ef, m * g
    b = 6 * math.pi * ETA_AIR * d / 2
    v = F / b
    fig, ax = plt.subplots(figsize=(9.5, 3.6))
    fig.subplots_adjust(left=0.27, right=0.97, top=0.78, bottom=0.2)
    ax.barh([1, 0], [F, W], left=1e-17, color=[CORAL, MINT]); ax.set_xscale("log"); ax.set_xlim(1e-17, 10 * max(F, W))
    ax.set_yticks([1, 0]); ax.set_yticklabels(["pull of the plates", "its weight"]); ax.set_xlabel("force (N)")
    spark = "  · the air would SPARK" if Ef > E_SPARK else ""
    fig.suptitle(f"E = {Ef:,.0f} N/C; pull = {F / W:,.0f} × its weight; drift {100 * v:.2f} cm/s{spark}",
                 x=0.03, ha="left", fontsize=14, color=GOLD)
    footnote(fig, "computed · air drag: Stokes's law, η = 1.82 × 10⁻⁵ Pa s"); plt.show()
    return Ef, F / W, v

E_h, ratio_h, v_h = plates_picture(2.0, 100)
assert round(E_h, -3) == 226000 and round(ratio_h, -2) == 700 and round(100 * v_h, 1) == 2.1

In [ ]:
interact(plates_picture, sigma_uC=FloatSlider(value=2.0, min=0.5, max=30.0, step=0.5, description="σ (µC/m²)"),
         electrons=IntSlider(value=100, min=10, max=500, step=10, description="electrons"));

## Sources
Every exercise and set-up here is our own. The constants are the course's rounded values (k, ε₀, e and the
electron's mass from CODATA 2022, physics.nist.gov/cuu/Constants); air's breakdown field, about 3 × 10⁶ V/m in dry air
at sea level, is a textbook value (Tipler); air's viscosity at 20 °C is from NBS Technical Note 1186. The simulator is
the course's: fourth-order Runge–Kutta with a fixed step.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib, sys
_src = pathlib.Path("../../../episodes/ph13_electric_fields/practice.py")
if _src.exists():
    sys.path.insert(0, str(_src.resolve().parents[2]))
    _spec = importlib.util.spec_from_file_location("ph13_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"], _q["parts"], _q["hint"]) == (
            _e["q"], _e["working"], _e["function"], _e["tier"], _e["parts"], _e["hint"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 * max(1, abs(b)) for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX)
    assert (K, EPS0, E, ME, g, E_SPARK, ETA_AIR) == (_P.K, _P.EPS0, _P.E, _P.ME, _P.g, _P.E_SPARK, _P.ETA_AIR)
    assert close(_P.HOOK["E"], E_h, 1e-12) and close(_P.HOOK["ratio"], ratio_h, 1e-9)
    print("matches the course's practice set")